In [4]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# Load dataset
possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv"
]
csv_path = next(p for p in possible_paths if os.path.exists(p))
df = pd.read_csv(csv_path)

# Label definition
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Precision@K Helper
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"Loaded {len(df):,} rows across {df['client_id'].nunique()} unique clients.")
print(f"Base Rate: {df['is_declining_label'].mean():.3f}")


Loaded 30,000 rows across 32 unique clients.
Base Rate: 0.542


# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Paper Finding 1: "Keyword search volume does not strongly predict traffic."

Methodology Question: Was the correlation measured across all historical windows, or only on active/visible pages? Filtering out zero-impression pages might reveal different non-linear thresholds for high-volume keywords.

Paper Finding 2: "Machine Learning models achieve a ~3x lift over rule baselines."

Methodology Question: Does this 3x lift hold up when testing on completely unseen client domains (grouped split), or does a portion of the lift stem from client-specific traffic patterns?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# Select features
feature_cols = [
    "impressions_90d", "clicks_90d", "avg_position", 
    "content_age_days", "days_since_last_update", 
    "word_count", "ctr"
]

X = df[feature_cols].fillna(df[feature_cols].median())
y = df["is_declining_label"]
groups = df["client_id"]

# 1. Random Split (Naive Validation)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42).fit(X_tr_r, y_tr_r)
prob_random = rf_random.predict_proba(X_te_r)[:, 1]
p50_random = precision_at_k(prob_random, y_te_r, k=50)

# 2. Grouped Split by Client (Honest Validation)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42).fit(X_tr_g, y_tr_g)
prob_grouped = rf_grouped.predict_proba(X_te_g)[:, 1]
p50_grouped = precision_at_k(prob_grouped, y_te_g, k=50)

# Display Comparison
split_summary = pd.DataFrame([
    {"Validation Design": "Random Split (Naive)", "Precision@50": round(p50_random, 3)},
    {"Validation Design": "Grouped-by-Client Split (Honest)", "Precision@50": round(p50_grouped, 3)},
    {"Validation Design": "Gap (Generalization Gap)", "Precision@50": round(p50_random - p50_grouped, 3)}
])

print("=== VALIDATION SPLIT AUDIT ===")
print(split_summary.to_string(index=False))


=== VALIDATION SPLIT AUDIT ===
               Validation Design  Precision@50
            Random Split (Naive)          0.86
Grouped-by-Client Split (Honest)          0.56
        Gap (Generalization Gap)          0.30


Split Interpretation:

The gap between Random Split and Grouped Split shows how much the model relied on memorizing client-level features versus learning general search decline patterns. The Grouped Split represents our true, honest evaluation on unseen client websites.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# Audit feature matrix for suspicious correlations
correlations = X.apply(lambda col: col.corr(y))
audit_df = pd.DataFrame({
    "Feature": feature_cols,
    "Correlation_with_Label": correlations.round(4)
}).sort_values("Correlation_with_Label", key=abs, ascending=False)

print("=== FEATURE LEAKAGE AUDIT ===")
print(audit_df.to_string(index=False))

# Check for perfect correlation (indicator of leakage)
max_corr = audit_df["Correlation_with_Label"].abs().max()
print(f"\nMax absolute correlation with target label: {max_corr:.4f}")
if max_corr < 0.85:
    print("STATUS: PASSED — No suspicious label-derived columns detected.")
else:
    print("STATUS: WARNING — Potential feature leakage detected!")


=== FEATURE LEAKAGE AUDIT ===
               Feature  Correlation_with_Label
      content_age_days                 -0.1639
            word_count                  0.0843
days_since_last_update                  0.0814
                   ctr                 -0.0619
            clicks_90d                 -0.0397
          avg_position                 -0.0290
       impressions_90d                 -0.0182

Max absolute correlation with target label: 0.1639
STATUS: PASSED — No suspicious label-derived columns detected.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Claim Rewrite Audit:

❌ Overpromising Claim (Before):
"Our Random Forest model accurately predicts Google's ranking algorithm and guarantees which pages will regain top positions."

✅ Safe, Rigorous Claim (After):

"In our client-grouped evaluation, the Random Forest model demonstrated a directional lift in Precision@50 over the hand-written baseline, serving as a decision-support tool to help human editors prioritize declining pages for content refresh."

Allowed Safe Vocabulary Used: observed, measured, directional, decision-support.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.